# 04 - Fine-Tuning Baseline

In questo notebook eseguiamo il fine-tuning di una ResNet-18 pre-addestrata sul dataset GTSRB.
A differenza dei primissimi esperimenti, dove scrivevamo i loop a mano, adottiamo **PyTorch Lightning** importando la logica dal nostro modulo `src/lightning_modules.py`.

Questo notebook rimane una **baseline locale**: usa una configurazione essenziale, senza Weights & Biases, senza pesi di classe, senza augmentation, senza label smoothing e senza scheduler. In questo modo possiamo verificare il comportamento del fine-tuning end-to-end prima di passare alla pipeline sperimentale completa del notebook successivo.

Per mantenere coerenza con la pipeline finale, usiamo la stessa barra di avanzamento `TQDMProgressBar` e rendiamo esplicito l'ottimizzatore `AdamW`, gia' previsto come default da `LitResNet`.


In [1]:
import sys
if '..' not in sys.path:
    sys.path.append('..')

import torch
import pytorch_lightning as pl
from pytorch_lightning.callbacks import TQDMProgressBar, ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger

from src.dataset import get_dataloaders
from src.lightning_modules import LitResNet
from src.utils import set_seed

# Impostiamo il seed per la riproducibilita' degli esperimenti.
set_seed(42)

# Ottimizzazione per Tensor Cores su GPU compatibili.
torch.set_float32_matmul_precision('medium')

## 1. Caricamento Dataset

Carichiamo i dataloader sfruttando la funzione unificata in `src/dataset.py`. Manteniamo `batch_size=512` e `num_workers=8`, gli stessi valori usati nella pipeline finale, ma impostiamo esplicitamente `augmentation_strategy="none"` per mantenere deterministici train, validation e test.


In [2]:
# Iperparametri
batch_size = 512
num_workers = 8

# Caricamento dei DataLoader: Lightning gestisce lo spostamento dei batch sul device.
train_loader, val_loader, test_loader = get_dataloaders(
    data_dir="../_data", 
    batch_size=batch_size, 
    num_workers=num_workers,
    augmentation_strategy="none",
)
print("Dati caricati correttamente.")

Dati caricati correttamente.


## 2. Definizione del Modello e Addestramento (Fine-tuning)

Istanziamo `LitResNet` con ResNet-18 preaddestrata, learning rate `1e-4` e ottimizzatore `AdamW`. Il training dura 30 epoche massime e resta locale: niente logger cloud, niente early stopping, niente pesi di classe, label smoothing disattivato e nessun scheduler. Le metriche finali sono calcolate sul checkpoint con F1 macro di validation migliore.

Nella run corretta, il checkpoint migliore e' stato trovato all'epoca 28 con F1 macro di validation `0.9978`. Sul test set ottiene accuracy `0.9347` e F1 macro `0.8906`: questo riferimento intenzionalmente essenziale permette di misurare il contributo della pipeline del notebook successivo.


In [3]:
# Inizializziamo il modulo Lightning per la baseline locale.
lit_model = LitResNet(
    num_classes=43,
    pretrained=True,
    lr=1e-4,
    optimizer_name="adamw",
    label_smoothing=0.0,
    use_onecycle_lr=False,
)

# Usiamo TQDM anche nella baseline per mantenere una visualizzazione coerente con la pipeline finale.
progress_bar = TQDMProgressBar(refresh_rate=10)

# Salviamo il checkpoint con il miglior F1-score macro di validazione.
checkpoint_callback = ModelCheckpoint(
    monitor="val_f1_score",
    mode="max",
    dirpath="checkpoints/baseline_fair",
    filename="resnet18-baseline-{epoch:02d}-{val_f1_score:.3f}",
    save_top_k=1,
)

# Logger CSV locale: salva le curve di loss/metriche in un file leggibile anche offline,
# usato dal notebook 05 per confrontare questa baseline con la pipeline finale.
# Usiamo una nuova versione per non mescolare le metriche della baseline corretta con la run storica.
csv_logger = CSVLogger(save_dir="logs", name="finetuning_baseline", version=1)

# Inizializziamo il Trainer senza logger WandB per una baseline locale.
trainer = pl.Trainer(
    max_epochs=30,
    accelerator="auto",
    logger=csv_logger,
    callbacks=[progress_bar, checkpoint_callback],
    log_every_n_steps=10
)

# Lanciamo l'addestramento.
print('Inizio fine-tuning della baseline in locale...')
trainer.fit(lit_model, train_dataloaders=train_loader, val_dataloaders=val_loader)

# Eseguiamo validation e test sul checkpoint con F1 macro migliore.
print('\nInizio calcolo metriche di Validation finali...')
trainer.validate(dataloaders=val_loader, ckpt_path="best")

# Esecuzione finale sul test set puro.
print('\nInizio test finale...')
trainer.test(dataloaders=test_loader, ckpt_path="best")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Inizio fine-tuning della baseline in locale...


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ ResNet             │ 11.2 M │ train │     0 │
│ 1 │ train_acc │ MulticlassAccuracy │      0 │ train │     0 │
│ 2 │ val_acc   │ MulticlassAccuracy │      0 │ train │     0 │
│ 3 │ test_acc  │ MulticlassAccuracy │      0 │ train │     0 │
│ 4 │ train_f1  │ MulticlassF1Score  │      0 │ train │     0 │
│ 5 │ val_f1    │ MulticlassF1Score  │      0 │ train │     0 │
│ 6 │ test_f1   │ MulticlassF1Score  │      0 │ train │     0 │
└───┴───────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 11.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 11.2 M                                                                                               
Total estimated model params size (MB): 44.794                                                                     
Modules in train mode: 74                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

Restoring states from the checkpoint path at C:\Users\franc\Desktop\Git_Hub_Deep_Learning-Applications_Francesco_Beraldi\DLA_LAB1\exercise_2_gtsrb_classification\checkpoints\baseline_fair\resnet18-baseline-epoch=28-val_f1_score=0.998.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at C:\Users\franc\Desktop\Git_Hub_Deep_Learning-Applications_Francesco_Beraldi\DLA_LAB1\exercise_2_gtsrb_classification\checkpoints\baseline_fair\resnet18-baseline-epoch=28-val_f1_score=0.998.ckpt



Inizio test finale...


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       test/accuracy       │    0.9346793293952942     │
│       test/f1_score       │    0.8905639052391052     │
│         test/loss         │    0.22684378921985626    │
└───────────────────────────┴───────────────────────────┘

[{'test/loss': 0.22684378921985626,
  'test/accuracy': 0.9346793293952942,
  'test/f1_score': 0.8905639052391052}]